# Agrupación automática de vehículos por características visuales
**Grupo 14 · Aprendizaje No Supervisado + Visión por Computadora**

Pipeline: fotos (Stanford Cars) → ResNet-50 (características) → PCA → K-Means → métricas, gráficos y prototipo.

> Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

## 1. Configuración

In [ ]:
import os, glob, random
import numpy as np

N_IMAGES = 3000     # cuántas fotos usar (None = todas; tarda más)
N_COMP = 50         # componentes de PCA
K_RANGE = range(2, 21)   # valores de K a probar
K_MANUAL = None     # pon un número (ej. 8) si quieres forzar K; None = el mejor por silueta
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
OUT = "resultados"
os.makedirs(OUT, exist_ok=True)

## 2. Descargar Stanford Cars (Kaggle)
Si `kagglehub` falla, descarga el dataset a mano desde Kaggle (*Stanford Cars Dataset*), súbelo a Colab y pon la carpeta en `path`.

In [ ]:
!pip -q install kagglehub
import kagglehub
path = kagglehub.dataset_download("jessicali9530/stanford-cars-dataset")
print("Dataset en:", path)

## 3. Listar las fotos y ver ejemplos (para el informe 5.3)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

rutas = sorted(glob.glob(os.path.join(path, "**", "*.jpg"), recursive=True))
print("Fotos encontradas:", len(rutas))
assert len(rutas) > 0, "No se encontraron fotos: revisa la variable path"

if N_IMAGES:
    rutas = random.sample(rutas, min(N_IMAGES, len(rutas)))
print("Fotos que se van a usar:", len(rutas))

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, r in zip(axs.ravel(), rutas[:8]):
    ax.imshow(Image.open(r).convert("RGB"))
    ax.axis("off")
plt.suptitle("Ejemplos de Stanford Cars")
plt.tight_layout()
plt.savefig(f"{OUT}/01_ejemplos.png", dpi=150)
plt.show()

## 4. Preprocesamiento y extracción de características con ResNet-50 (informe 5.3 y 5.4)
Cada foto se ajusta a 224×224, se normaliza y pasa por ResNet-50 sin su última capa → un vector de **2048 números**.

In [ ]:
import torch
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

prep = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class Fotos(Dataset):
    def __init__(self, rutas):
        self.rutas = rutas
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        return prep(Image.open(self.rutas[i]).convert("RGB"))

modelo = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
modelo.fc = torch.nn.Identity()      # quitamos la capa de clasificación
modelo = modelo.to(device).eval()

loader = DataLoader(Fotos(rutas), batch_size=64, num_workers=2)
lotes = []
with torch.no_grad():
    for lote in loader:
        lotes.append(modelo(lote.to(device)).cpu().numpy())

features = np.vstack(lotes)
print("Tamaño de features:", features.shape)   # (N, 2048)
np.save(f"{OUT}/features.npy", features)

## 5. Reducción de dimensionalidad con PCA (informe 5.5)

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=SEED).fit(features)
acum = np.cumsum(pca_full.explained_variance_ratio_)

pca = PCA(n_components=N_COMP, random_state=SEED)
X = pca.fit_transform(features)
print(f"{N_COMP} componentes conservan {acum[N_COMP-1]*100:.1f}% de la varianza")

plt.figure(figsize=(7, 4))
plt.plot(range(1, 201), acum[:200])
plt.axvline(N_COMP, color="red", ls="--", label=f"{N_COMP} componentes")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("PCA sobre las características de ResNet-50")
plt.legend()
plt.grid(alpha=.3)
plt.savefig(f"{OUT}/02_pca_varianza.png", dpi=150)
plt.show()

## 6. K-Means: elegir K con el método del codo y la silueta (informe 5.6)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = list(K_RANGE)
inercias, siluetas = [], []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    inercias.append(km_k.inertia_)
    siluetas.append(silhouette_score(X, km_k.labels_, sample_size=min(5000, len(X)), random_state=SEED))
    print(f"K={k:2d}  inercia={km_k.inertia_:,.0f}  silueta={siluetas[-1]:.3f}")

K_mejor = ks[int(np.argmax(siluetas))]
print("\nK con mejor silueta:", K_mejor)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(ks, inercias, "o-")
axs[0].set_title("Método del codo")
axs[0].set_xlabel("K"); axs[0].set_ylabel("Inercia")
axs[1].plot(ks, siluetas, "o-", color="green")
axs[1].axvline(K_mejor, color="red", ls="--")
axs[1].set_title("Silhouette score según K")
axs[1].set_xlabel("K"); axs[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.savefig(f"{OUT}/03_codo_silueta.png", dpi=150)
plt.show()

**Ojo:** en fotos reales la silueta suele ser más alta con K pequeño (2 o 3). Si pasa eso, mira el codo y decide un K más útil
(por ejemplo 8 o 10) poniéndolo en `K_MANUAL` arriba, y justifícalo en el informe.

## 7. Agrupar y evaluar con métricas (informe 5.7)

In [ ]:
import pandas as pd
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

K = K_MANUAL or K_mejor
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
etiquetas = km.labels_
np.save(f"{OUT}/etiquetas.npy", etiquetas)

tabla = pd.DataFrame({
    "Métrica": ["K (número de grupos)", "Silhouette (más alto es mejor)",
                "Davies-Bouldin (más bajo es mejor)", "Calinski-Harabasz (más alto es mejor)"],
    "Valor": [K,
              round(silhouette_score(X, etiquetas, sample_size=min(5000, len(X)), random_state=SEED), 4),
              round(davies_bouldin_score(X, etiquetas), 4),
              round(calinski_harabasz_score(X, etiquetas), 1)],
})
print(tabla.to_string(index=False))
tabla.to_csv(f"{OUT}/04_metricas.csv", index=False)

print("\nFotos por grupo:")
print(pd.Series(etiquetas).value_counts().sort_index().to_string())

## 8. Visualización de los grupos (informe 5.8)

In [ ]:
# 8.1 Gráfico 2D de los grupos (primeras 2 componentes de PCA)
plt.figure(figsize=(8, 6))
sc = plt.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=8)
plt.legend(*sc.legend_elements(), title="Grupo", loc="best", fontsize=8)
plt.title(f"Grupos de vehículos (K-Means, K={K}) en 2 componentes de PCA")
plt.xlabel("Componente 1"); plt.ylabel("Componente 2")
plt.savefig(f"{OUT}/05_grupos_2d.png", dpi=150)
plt.show()

# 8.2 Mosaico: 8 fotos de cada grupo
n_por_grupo = 8
fig, axs = plt.subplots(K, n_por_grupo, figsize=(2 * n_por_grupo, 1.8 * K))
axs = np.atleast_2d(axs)
for g in range(K):
    idx = np.where(etiquetas == g)[0][:n_por_grupo]
    for j in range(n_por_grupo):
        ax = axs[g, j]
        ax.axis("off")
        if j < len(idx):
            ax.imshow(Image.open(rutas[idx[j]]).convert("RGB"))
        if j == 0:
            ax.set_title(f"Grupo {g}", fontsize=9, loc="left")
plt.tight_layout()
plt.savefig(f"{OUT}/06_mosaico_grupos.png", dpi=120)
plt.show()

## 9. Guardar el modelo y probar el prototipo (informe 5.9)

In [ ]:
import joblib
joblib.dump(pca, f"{OUT}/pca.joblib")
joblib.dump(km, f"{OUT}/kmeans.joblib")

!pip -q install gradio
import gradio as gr
from sklearn.neighbors import NearestNeighbors

vecinos = NearestNeighbors(n_neighbors=5).fit(X)

def agrupar(img):
    foto = Image.fromarray(img).convert("RGB")
    x = prep(foto).unsqueeze(0).to(device)
    with torch.no_grad():
        f = modelo(x).cpu().numpy()
    z = pca.transform(f)
    grupo = int(km.predict(z)[0])
    _, idx = vecinos.kneighbors(z)
    similares = [Image.open(rutas[i]).convert("RGB") for i in idx[0]]
    return f"Grupo {grupo}", similares

demo = gr.Interface(
    fn=agrupar,
    inputs=gr.Image(type="numpy", label="Sube la foto de un vehículo"),
    outputs=[gr.Textbox(label="Grupo asignado"), gr.Gallery(label="Vehículos más parecidos")],
    title="Agrupación automática de vehículos",
    description="ResNet-50 + PCA + K-Means · Grupo 14",
)
demo.launch(share=True)